# Camargo BPIC17 — FullShared + n-gram=15 loader

Sibling of `train_camargo_LSTM.ipynb`. The original notebook reused the U-ED-LSTM encoder-decoder windowing, which gives the Camargo next-event predictor mostly EOS targets and no signal at short prefixes. This variant swaps in:

1. **`CamargoNGramDataset`** — paper-faithful sequence creation (Camargo et al. §3.1 / Table 1). For each case of length L we emit L samples, one per time-step, with a fixed n-gram window and the actual next activity as target.
2. **n-gram size 15** — the paper (Table 3) doesn't cover BPIC17 directly, but its closest analogue (BPI 2012: 36 activities, complex sequence flow) uses n-gram size 15. BPIC17 is similarly complex (28 activities, long cases).
3. **Feature projection** — the base pickle carries 8+ categoricals and 9 numericals, but the Camargo model + interpretability pipeline only consume `concept:name` + `org:resource` + `case_elapsed_time`. The loader projects down to `cat_indices=(0, 2)` + `num_indices=(0,)` before emitting samples.
4. **lr=1e-3** (paper-aligned Adam default). The earlier `lr=1e-6` was inherited from the U-ED-LSTM setup and is hundreds of times too small.

Architecture stays `FullShared_Join_LSTM` — there is no role-grouped BPIC17 pickle, and the paper's architecture ablation for BPIC17 is not reported. Checkpoint written to a fresh filename so the legacy artifact stays available for comparison.

# Imports

In [1]:
import importlib
import sys
sys.path.insert(0, '../../../../../../..')  # -> src/
import torch


# Data

In [2]:
import sys
sys.path.insert(0, '../../../../../../..')  # -> src/
file_path_train = '../../Loader/pkl/BPIC_2017_all_5_train.pkl'
bpic17_train_dataset = torch.load(file_path_train, weights_only=False)
print(type(bpic17_train_dataset), 'train len:', len(bpic17_train_dataset))

file_path_val = '../../Loader/pkl/BPIC_2017_all_5_val.pkl'
bpic17_val_dataset = torch.load(file_path_val, weights_only=False)
print(type(bpic17_val_dataset), 'val len:', len(bpic17_val_dataset))

<class 'event_log_loader.new_event_log_loader.EventLogDataset'> train len: 820824
<class 'event_log_loader.new_event_log_loader.EventLogDataset'> val len: 191265


In [3]:
import sys
sys.path.insert(0, '../../../../../../..')  # -> src/
bpic17_all_categories = bpic17_train_dataset.all_categories
bpic17_all_categories_cat = bpic17_all_categories[0]
bpic17_all_categories_num = bpic17_all_categories[1]

for i, cat in enumerate(bpic17_all_categories_cat):
    print(f'Cat[{i}]: {cat[0]:25s} #classes={cat[1]}')
for i, num in enumerate(bpic17_all_categories_num):
    print(f'Num[{i}]: {num[0]}')

concept_name = 'concept:name'
concept_name_id = [i for i, cat in enumerate(bpic17_all_categories_cat) if cat[0] == concept_name][0]
concept_name_size = bpic17_all_categories_cat[concept_name_id][1]
eos_id = bpic17_all_categories_cat[concept_name_id][2]['EOS']
print(f'\nactivity idx={concept_name_id}, vocab size={concept_name_size}, EOS id={eos_id}')

# Project to the Camargo subset: concept:name + org:resource for cats, case_elapsed_time for nums.
# Matches `_camargo_utils.BPIC17.cat_indices` = (0, 2) and `num_indices=(0,)`.
resource_id = [i for i, cat in enumerate(bpic17_all_categories_cat) if cat[0] == 'org:resource'][0]
CAT_INDICES = (concept_name_id, resource_id)
NUM_INDICES = (0,)
print(f'projection: cat_indices={CAT_INDICES}, num_indices={NUM_INDICES}')

Cat[0]: concept:name              #classes=28
Cat[1]: Action                    #classes=7
Cat[2]: org:resource              #classes=151
Cat[3]: EventOrigin               #classes=5
Cat[4]: lifecycle:transition      #classes=9
Cat[5]: case:LoanGoal             #classes=16
Cat[6]: case:ApplicationType      #classes=4
Cat[7]: Accepted                  #classes=5
Cat[8]: Selected                  #classes=5
Num[0]: case_elapsed_time
Num[1]: event_elapsed_time
Num[2]: day_in_week
Num[3]: seconds_in_day
Num[4]: case:RequestedAmount
Num[5]: FirstWithdrawalAmount
Num[6]: NumberOfTerms
Num[7]: MonthlyCost
Num[8]: CreditScore

activity idx=0, vocab size=28, EOS id=11
projection: cat_indices=(0, 2), num_indices=(0,)


In [4]:
import sys
sys.path.insert(0, '../../../../../../..')  # -> src/
# Build the `model_feat` list restricted to the projected subset — the model's
# embedding layer count must match what the loader will feed it.
model_feat_cat = [bpic17_all_categories_cat[i][0] for i in CAT_INDICES]
model_feat_num = [bpic17_all_categories_num[i][0] for i in NUM_INDICES]
model_feat = [model_feat_cat, model_feat_num]
print('model_feat:', model_feat)

# Project `data_set_categories` to match — required so the model sizes
# its embeddings correctly and saves/loads cleanly.
projected_categories = (
    [bpic17_all_categories_cat[i] for i in CAT_INDICES],
    [bpic17_all_categories_num[i] for i in NUM_INDICES],
)

model_feat: [['concept:name', 'org:resource'], ['case_elapsed_time']]


# Model

In [5]:
import sys
sys.path.insert(0, '../../../../../../..')  # -> src/
import joinLSTM.model
importlib.reload(joinLSTM.model)
from joinLSTM.model import FullShared_Join_LSTM

hidden_size = 50
num_layers = 1
input_size = 1  # sentinel -> model computes it from embeddings

model = FullShared_Join_LSTM(
    data_set_categories=projected_categories,
    hidden_size=hidden_size,
    num_layers=num_layers,
    model_feat=model_feat,
    input_size=input_size,
    output_size_act=concept_name_size,
)

Data set categories:  ([('concept:name', 28, {'A_Accepted': 1, 'A_Cancelled': 2, 'A_Complete': 3, 'A_Concept': 4, 'A_Create Application': 5, 'A_Denied': 6, 'A_Incomplete': 7, 'A_Pending': 8, 'A_Submitted': 9, 'A_Validating': 10, 'EOS': 11, 'O_Accepted': 12, 'O_Cancelled': 13, 'O_Create Offer': 14, 'O_Created': 15, 'O_Refused': 16, 'O_Returned': 17, 'O_Sent (mail and online)': 18, 'O_Sent (online only)': 19, 'W_Assess potential fraud': 20, 'W_Call after offers': 21, 'W_Call incomplete files': 22, 'W_Complete application': 23, 'W_Handle leads': 24, 'W_Personal Loan collection': 25, 'W_Shortened completion ': 26, 'W_Validate application': 27}), ('org:resource', 151, {'EOS': 1, 'User_1': 2, 'User_10': 3, 'User_100': 4, 'User_101': 5, 'User_102': 6, 'User_103': 7, 'User_104': 8, 'User_105': 9, 'User_106': 10, 'User_107': 11, 'User_108': 12, 'User_109': 13, 'User_11': 14, 'User_110': 15, 'User_111': 16, 'User_112': 17, 'User_113': 18, 'User_114': 19, 'User_115': 20, 'User_116': 21, 'User_117

/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


# Training

In [6]:
import sys
sys.path.insert(0, '../../../../../../..')  # -> src/
import training.camargo_ngram_dataset
import training.train_ngram
importlib.reload(training.camargo_ngram_dataset)
importlib.reload(training.train_ngram)
from training.camargo_ngram_dataset import CamargoNGramDataset
from training.train_ngram import NGramTraining

from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.tensorboard import SummaryWriter

NGRAM_SIZE = 15

ngram_train = CamargoNGramDataset(
    bpic17_train_dataset, ngram_size=NGRAM_SIZE,
    activity_idx=concept_name_id, eos_idx=eos_id,
    cat_indices=CAT_INDICES, num_indices=NUM_INDICES,
)
ngram_val = CamargoNGramDataset(
    bpic17_val_dataset, ngram_size=NGRAM_SIZE,
    activity_idx=concept_name_id, eos_idx=eos_id,
    cat_indices=CAT_INDICES, num_indices=NUM_INDICES,
)
print(f'n-gram train: {len(ngram_train)} samples (from {len(bpic17_train_dataset)} base windows)')
print(f'n-gram val:   {len(ngram_val)} samples (from {len(bpic17_val_dataset)} base windows)')

writer = SummaryWriter(comment='Full_bpic17_camargo_ngram15')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

learning_rate = 1e-3
optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate, weight_decay=0)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=4, min_lr=1e-10)

num_epochs = 100
batch_size = 256
shuffle = True

optimize_values = {
    'optimizer': optimizer,
    'scheduler': scheduler,
    'epochs': num_epochs,
    'mini_batches': batch_size,
    'shuffle': shuffle,
}

trainer = NGramTraining(
    model=model,
    device=device,
    data_train=ngram_train,
    data_val=ngram_val,
    optimize_values=optimize_values,
    writer=writer,
    save_model_n_th_epoch=1,
    saving_path='../pkl/BPIC17_camargo_ngram15.pkl',
)

trainer.train()

n-gram train: 797844 samples (from 820824 base windows)
n-gram val:   185964 samples (from 191265 base windows)
Device:  cpu
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
Scheduler: <torch.optim.lr_scheduler.ReduceLROnPlateau object at 0x139575310>
Epochs: 100  Mini-batch: 256  Shuffle: True


  0%|          | 0/100 [00:00<?, ?it/s]

Epoch [1/100], LR: 0.001
Training:   Avg Loss: 0.4235
Validation: Avg Loss: 0.2805
saving model
Epoch [2/100], LR: 0.001
Training:   Avg Loss: 0.2693
Validation: Avg Loss: 0.2670
saving model
Epoch [3/100], LR: 0.001
Training:   Avg Loss: 0.2590
Validation: Avg Loss: 0.2608
saving model
Epoch [4/100], LR: 0.001
Training:   Avg Loss: 0.2538
Validation: Avg Loss: 0.2554
saving model
Epoch [5/100], LR: 0.001
Training:   Avg Loss: 0.2505
Validation: Avg Loss: 0.2545
saving model
Epoch [6/100], LR: 0.001
Training:   Avg Loss: 0.2482
Validation: Avg Loss: 0.2549
saving model
Epoch [7/100], LR: 0.001
Training:   Avg Loss: 0.2462
Validation: Avg Loss: 0.2538
saving model
Epoch [8/100], LR: 0.001
Training:   Avg Loss: 0.2447
Validation: Avg Loss: 0.2527
saving model
Epoch [9/100], LR: 0.001
Training:   Avg Loss: 0.2432
Validation: Avg Loss: 0.2533
saving model
Epoch [10/100], LR: 0.001
Training:   Avg Loss: 0.2419
Validation: Avg Loss: 0.2518
saving model
Epoch [11/100], LR: 0.001
Training:   A

KeyboardInterrupt: 